In [1]:
# import important libaries
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder , LabelEncoder
from sklearn.compose import ColumnTransformer

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

In [3]:
# read data from local file
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\covid_toy.csv")
df 

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [4]:
df["fever"].fillna(df["fever"].mean(),inplace = True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7184\3056235766.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["fever"].fillna(df["fever"].mean(),inplace = True)


In [5]:
# make a target columns
X = df.drop("has_covid", axis=1)
y = df["has_covid"]

In [6]:
# Split into Training and Testing
cat = X.select_dtypes(include=["object"]).columns
num = X.select_dtypes(exclude=["object"]).columns

In [7]:
# pretranform data 
pre = ColumnTransformer(
    transformers=[
        ("cat" , OneHotEncoder(drop= "first" , handle_unknown="ignore" ), cat),
        ("num", StandardScaler(),num)
    ]
)
pre

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [8]:
# Split into Training and Testing
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y ,test_size=0.2, random_state=42)

In [9]:
# Fitting data into preprocess
X_train = pre.fit_transform(X_train)
X_test = pre.transform(X_test)

In [10]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [11]:
model = Sequential()
# Input Layer
# model.add(Input(shape=(X_train.shape[1],)))

# Hidden Layer 1
model.add(Dense(32, input_dim =X_train.shape[1], activation='relu'))

# Hidden Layer 2
model.add(Dense(16, activation='relu'))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
sgd = SGD(
    learning_rate=0.01,
    momentum=0.9,
    nesterov=True
)

model.compile(
    optimizer=sgd,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [29]:
model.fit(X_train , y_train , epochs=40 , batch_size=20 , verbose=1 , validation_data= (X_test , y_test), validation_split=0.2)

Epoch 1/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 166ms/step - accuracy: 0.8875 - loss: 0.3406 - val_accuracy: 0.5000 - val_loss: 0.7462
Epoch 2/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8750 - loss: 0.3389 - val_accuracy: 0.5000 - val_loss: 0.7616
Epoch 3/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.9000 - loss: 0.3395 - val_accuracy: 0.5000 - val_loss: 0.7592
Epoch 4/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.9000 - loss: 0.3362 - val_accuracy: 0.5000 - val_loss: 0.7786
Epoch 5/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.8625 - loss: 0.3341 - val_accuracy: 0.5000 - val_loss: 0.7706
Epoch 6/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9000 - loss: 0.3308 - val_accuracy: 0.5000 - val_loss: 0.7880
Epoch 7/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.9000 - loss: 0.3315 - val_accuracy: 0.5000 - val_loss: 0.7782
Epoch 8/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9000 - loss: 0.3242 - val_accuracy: 0.5000 - val_loss: 0.7883

In [30]:
# check accuarcy of model
loss, accuracy = model.evaluate(X_train, y_train)

print(f"Loss: {loss}")
print(f"Accuracy: {accuracy * 100:.2f}%")

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.9625 - loss: 0.2355 
Loss: 0.2354615181684494
Accuracy: 96.25%
